# Chấm caption đã lưu — CONTROL10, PROMPT9 và PROMPT10
**Accelerator None, Internet ON. Không cần Add Input hoặc API Secret.** Không model/checkpoint/GPU, không sinh lại caption, không TEST hoặc train.
Đọc bộ caption5000VALID đã khóa bằngSHA từ đúng hai reviewZIP, giữ nguyênreferencePTB của lượttrain. CIDEr phải replay lệch≤1e-7 trước khi nhận kết quả; không nới tolerance. Sau đó tínhBLEU4/ROUGE-L/CHAIRs/CHAIRi/Recall vàpairedCI.
CONTROLbest10 vàPROMPTbest9 đã khóa bằngCIDEr trướcCHAIR. PROMPT10 chỉ là endpointcùng10epoch; không tự chọn lại theoCHAIR. CHAIR chỉ chấm đối tượng, không phải độ đúng quan hệ/thuộc tính. One seed/reusedVALID, chưa là kết quảTEST độc lập.
Chạy Save Version → Save & Run All vớiCPU và gửi `csa_prompt_scratch_quality_valid5000_review.zip`. Cap1h là giới hạn dừng, không phảiETA.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile
from pathlib import Path
OUTPUT=Path('/kaggle/working/csa_prompt_scratch_quality_valid5000')
if OUTPUT.exists():raise RuntimeError('Preserve prior output; use fresh path/session')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='prompt-scratch-quality-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
DEADLINE=time.monotonic()+3600 # CPU STOPcap, not ETA.
REVISION='17bf52c3f17c7b44e67f873d2405dcda2bb55a92'
HASHES={'score_prompt_scratch_quality.py': 'd739c7bed3faf1a91112c4ef9f89c2f23b1aa098ad6b1e8bf51f9cd4baa783e7', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'test_prompt_scratch_quality_data.py': '465c0cd5278453a2c09d775ab78bf566960e039ffbf54a8fa594c0bd3baa48f3', 'make_prompt_scratch_quality_notebook.py': '69ca70ecf91a8f95f020b7a6f23914398a6f51bf271092dcddbce0fc2c95abe5', 'prompt_scratch_valid5000_saved.json.gz': 'e8eb7623eaad1c36bf5d813e7ea3f9d9dd45fa4bce055d378f3e9b1f5d96555a', 'PROMPT_SCRATCH_QUALITY_INPUTS.json': '58915136419682ad051d6bd34cf576293e49655c4166de37c1b3a00e17c853a9'}
for n,h in HASHES.items():
    raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/experiments/csa_paper/'+n,timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Changed source/payload:'+n)
    (SOURCE/n).write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    timeout=min(timeout,max(.01,DEADLINE-time.monotonic()))
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CPU saved-caption metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('CPU metric cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_prompt_scratch_quality_data.py'],timeout=120)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    CHAIR=SOURCE/'chair'
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    if hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()!='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':raise ValueError('Changed CHAIR cache')
    run([sys.executable,'-u',SOURCE/'score_prompt_scratch_quality.py','--payload',SOURCE/'prompt_scratch_valid5000_saved.json.gz',
        '--manifest',SOURCE/'PROMPT_SCRATCH_QUALITY_INPUTS.json','--chair',CHAIR,'--output',OUTPUT],timeout=1800)
    result=json.loads((OUTPUT/'comparison.json').read_text(encoding='utf-8'))
    complete=result['status']=='PROMPT_SCRATCH_QUALITY_COMPLETE' and len(result['metrics'])==3
    if not complete:raise RuntimeError('Incomplete quality evaluation')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,model_executed=False,new_generation=False,test_used=False)),encoding='utf-8')
    REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file():z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',REVIEW,flush=True)
    print('QUALITY COMPLETE' if complete else 'INCOMPLETE; preserve diagnostics; no partial-model ranking.',flush=True)
